# Trabajo en Clase

### Sesiones 1 y 2 · Práctica en Databricks

Arquitecturas ETL y Modelos de IA Aplicados · EAM · 2026-II
Robert García Rey

En esta práctica crearemos un notebook desde cero para conocer el entorno de Databricks, cargar archivos y trabajar con SQL y PySpark. Consultaremos tablas, aplicaremos operaciones Delta y crearemos una vista y una función.

#### Desarrollo

Establece analitica.data como el contexto de trabajo para ejecutar las siguientes consultas SQL sobre sus tablas.

In [0]:
%sql
USE analitica.data

Crea o reemplaza la tabla tabla_simple con dos columnas: id de tipo entero y nombre de tipo texto. La tabla utiliza el formato Delta y, si ya existe, reemplaza su contenido y estructura.

In [0]:
%sql
CREATE OR REPLACE TABLE tabla_simple(
    id INT,
    nombre STRING
)

USING DELTA

Inserta tres registros en la tabla tabla_simple, con los IDs 1, 2 y 3, y los nombres Ana, Carlos y María.

In [0]:
%sql
INSERT INTO tabla_simple (id, nombre)
VALUES
    (1, 'Ana'),
    (2, 'Carlos'),
    (3, 'María');

num_affected_rows,num_inserted_rows
3,3


Consulta y muestra todos los registros y columnas de la tabla tabla_simple.

In [0]:
%sql
SELECT * 
FROM tabla_simple;

id,nombre
1,Ana
2,Carlos
3,María


Importa las funciones de PySpark con el alias F y guarda en la variable ruta la ubicación /volumenes/analitica/data/landing, donde se encuentran o se almacenarán archivos.

In [0]:
from pyspark.sql import functions as F
ruta= "/volumenes/analitica/data/landing"

Crea o reemplaza la función fn_mayor_que_10, que recibe un número decimal y devuelve TRUE si es mayor que 10 o FALSE si es menor o igual a 10.

In [0]:

%sql
CREATE OR REPLACE FUNCTION analitica.data.fn_mayor_que_10(x DOUBLE)
RETURNS BOOLEAN
RETURN x > 10;


Consulta todos los registros de tabla_simple y devuelve únicamente las filas si la función fn_mayor_que_10(11) es verdadera. Como 11 es mayor que 10, se muestran todos los registros de la tabla.

In [0]:
%sql
SELECT * FROM tabla_simple 
WHERE analitica.data.fn_mayor_que_10(11)

id,nombre
1,Ana
2,Carlos
3,María


Lee el archivo data_demo.csv desde la ruta indicada, detecta automáticamente los tipos de datos y utiliza la primera fila como encabezado. Luego muestra las primeras 10 filas del DataFrame df.

In [0]:

path_data_demo = "/Volumes/analitica/data/landing/data_demo.csv"

df = spark.read.csv(
    path=path_data_demo,
    header=True,
    inferSchema=True
)

display(df.limit(10))


rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0
3.0,2.0,180.0,1131950.0,5,154.0,0
3.0,1.0,67.0,452672.0,4,15.0,0
3.0,3.0,198.0,3116290.0,3,306.3557294709182,0
4.0,2.0,245.0,1202560.0,3,24.824464122611424,0
2.0,2.0,66.0,419282.0,5,24.0,0
2.0,1.0,114.0,1830260.0,1,251.39533835163672,0
2.0,1.0,56.0,274005.0,5,93.0,0


Guarda el DataFrame df como una tabla Delta llamada inmuebles. Si la tabla ya existe, sobrescribe sus datos actuales.

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("inmuebles")

Carga la tabla inmuebles en el DataFrame df_inmuebles y muestra sus primeras 3 filas.

In [0]:
df_inmuebles = spark.table("inmuebles")
display(df_inmuebles.limit(3))

rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0


Consulta y muestra las primeras 5 filas de la tabla inmuebles con todas sus columnas.

In [0]:
%sql
SELECT * FROM inmuebles LIMIT 5

rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0
3.0,2.0,180.0,1131950.0,5,154.0,0
3.0,1.0,67.0,452672.0,4,15.0,0


Lee el archivo spark_intro.csv, toma la primera fila como encabezado, infiere automáticamente los tipos de datos y muestra las primeras 10 filas en el DataFrame df_pedidos.

In [0]:

path_data_demo = "/Volumes/analitica/data/landing/spark_intro.csv"

df_pedidos = spark.read.csv(
    path=path_data_demo,
    header=True,
    inferSchema=True
)

display(df_pedidos.limit(10))


order_id,order_date,country,product,category,units_sold,unit_price
1,2024-01-02,Spain,Laptop,Electronics,2,1200
2,2024-01-03,Spain,Mouse,Accessories,5,25
3,2024-01-05,France,Laptop,Electronics,1,1150
4,2024-01-06,Germany,Keyboard,Accessories,3,45
5,2024-01-07,Spain,Monitor,Electronics,2,300
6,2024-01-08,France,Mouse,Accessories,10,23
7,2024-01-10,Germany,Laptop,Electronics,1,1300
8,2024-01-11,Spain,Keyboard,Accessories,4,42
9,2024-01-12,France,Monitor,Electronics,1,280
10,2024-01-14,Germany,Mouse,Accessories,6,27


Importa las funciones de PySpark con el alias F para utilizarlas en operaciones de transformación y análisis de datos.

In [0]:
from pyspark.sql import functions as F

Selecciona las columnas order_id, country, product y units_sold del DataFrame df_pedidos y filtra los registros para mostrar únicamente los pedidos de España.

In [0]:
df_pedidos.select("order_id","country","product","units_sold").filter(F.col("country")== "Spain")

DataFrame[order_id: int, country: string, product: string, units_sold: int]

Crea la columna importe multiplicando las unidades vendidas (units_sold) por el precio unitario (unit_price) y muestra el DataFrame completo con los importes calculados.

In [0]:
df_importes = df_pedidos.withColumn(
    "importe",
    F.col("units_sold") * F.col("unit_price")
)

display(df_importes)

order_id,order_date,country,product,category,units_sold,unit_price,importe
1,2024-01-02,Spain,Laptop,Electronics,2,1200,2400
2,2024-01-03,Spain,Mouse,Accessories,5,25,125
3,2024-01-05,France,Laptop,Electronics,1,1150,1150
4,2024-01-06,Germany,Keyboard,Accessories,3,45,135
5,2024-01-07,Spain,Monitor,Electronics,2,300,600
6,2024-01-08,France,Mouse,Accessories,10,23,230
7,2024-01-10,Germany,Laptop,Electronics,1,1300,1300
8,2024-01-11,Spain,Keyboard,Accessories,4,42,168
9,2024-01-12,France,Monitor,Electronics,1,280,280
10,2024-01-14,Germany,Mouse,Accessories,6,27,162


Calcula la suma de todos los valores de la columna importe para obtener las ventas totales y muestra el resultado con el nombre ventas_totales.

In [0]:
df_importes.agg(
    F.sum("importe").alias("ventas_totales")
).show()

+--------------+
|ventas_totales|
+--------------+
|          6550|
+--------------+



Agrupa los pedidos por país, suma los importes de cada grupo para calcular las ventas totales por país y muestra los resultados.

In [0]:
df_importes.groupBy("country").agg(
    F.sum("importe").alias("ventas_totales")
).show()

+-------+--------------+
|country|ventas_totales|
+-------+--------------+
|  Spain|          3293|
| France|          1660|
|Germany|          1597|
+-------+--------------+



Agrupa los pedidos por producto, calcula las ventas totales de cada uno y ordena los resultados de mayor a menor según las ventas.

In [0]:
df_importes.groupBy("product").agg(
    F.sum("importe").alias("ventas_totales")
).orderBy(
    F.desc("ventas_totales")
).show()

+--------+--------------+
| product|ventas_totales|
+--------+--------------+
|  Laptop|          4850|
| Monitor|           880|
|   Mouse|           517|
|Keyboard|           303|
+--------+--------------+



Agrupa los pedidos por país, calcula el total de unidades vendidas y el importe total redondeado a dos decimales, y muestra los resultados ordenados de mayor a menor importe total.

In [0]:
resumen_pais = df_importes.groupBy("country").agg(
    F.sum("units_sold").alias("unidades"),
    F.round(F.sum("importe"), 2).alias("importe_total")
)

display(resumen_pais.orderBy(F.desc("importe_total")))

country,unidades,importe_total
Spain,13,3293
France,12,1660
Germany,10,1597


Lee el archivo dim_spark_intro.csv definiendo las columnas product, product_family y supplier como texto. Si encuentra datos con formato inválido, detiene la lectura y muestra el DataFrame df_productos.

In [0]:
path_file = "/Volumes/analitica/data/landing/dim_spark_intro.csv"


df_productos = spark.read.csv(
    path=path_file,
    header=True,
    schema= "product STRING, product_family STRING, supplier STRING",
    mode="FAILFAST"
)

display(df_productos)


product,product_family,supplier
Laptop,Computers,TechCorp
Mouse,Peripherals,InputMakers
Keyboard,Peripherals,InputMakers
Monitor,Displays,ScreenWorld


Agrupa los pedidos por producto, cuenta cuántas veces aparece cada uno y muestra únicamente los productos que aparecen más de una vez.

In [0]:
display(
    df_pedidos
    .groupBy("product")
    .count()
    .filter("count > 1")
)

product,count
Mouse,3
Monitor,2
Keyboard,2
Laptop,3


Combina df_importes con df_productos mediante la columna product, conservando todos los pedidos y agregando la información de familia y proveedor cuando exista coincidencia. Muestra el DataFrame resultante df_enriquecido.

In [0]:
df_enriquecido = df_importes.join(df_productos,"product", "left")

display(df_enriquecido)

product,order_id,order_date,country,category,units_sold,unit_price,importe,product_family,supplier
Laptop,1,2024-01-02,Spain,Electronics,2,1200,2400,Computers,TechCorp
Mouse,2,2024-01-03,Spain,Accessories,5,25,125,Peripherals,InputMakers
Laptop,3,2024-01-05,France,Electronics,1,1150,1150,Computers,TechCorp
Keyboard,4,2024-01-06,Germany,Accessories,3,45,135,Peripherals,InputMakers
Monitor,5,2024-01-07,Spain,Electronics,2,300,600,Displays,ScreenWorld
Mouse,6,2024-01-08,France,Accessories,10,23,230,Peripherals,InputMakers
Laptop,7,2024-01-10,Germany,Electronics,1,1300,1300,Computers,TechCorp
Keyboard,8,2024-01-11,Spain,Accessories,4,42,168,Peripherals,InputMakers
Monitor,9,2024-01-12,France,Electronics,1,280,280,Displays,ScreenWorld
Mouse,10,2024-01-14,Germany,Accessories,6,27,162,Peripherals,InputMakers


Cuenta los registros antes y después de la unión y verifica si ambas cantidades son iguales; si no coinciden, indica que se debe revisar la llave product por posibles duplicados.

In [0]:
print("pedidos antes: ", df_pedidos.count())
print("pedidos despues: ", df_enriquecido.count())
display(df_enriquecido.count()== df_pedidos.count(),"revisar la llave por producto")

pedidos antes:  10
pedidos despues:  10


True

'revisar la llave por producto'

Guarda el DataFrame df_enriquecido como una tabla Delta llamada pedidos_enriquecido, sobrescribiendo los datos existentes si la tabla ya existe.

In [0]:
df_enriquecido.write.format("delta").mode("overwrite").saveAsTable("pedidos_enriquecido")


Lee el archivo ventas_2025.csv definiendo las columnas id, fecha, producto, cantidad y precio con sus tipos de datos correspondientes. Si encuentra datos inválidos, detiene la lectura y muestra el DataFrame df_ventas.

In [0]:
path_file = "/Volumes/analitica/data/landing/ventas_2025.csv"



df_ventas = spark.read.csv(
    path=path_file,
    header=True,
    schema= "id INT, fecha DATE, producto STRING, cantidad INT, precio DOUBLE",
    mode="FAILFAST"
)

display(df_ventas)


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


Verifica el esquema y la cantidad de ventas, identifica IDs duplicados y detecta registros inválidos por valores nulos, precios no numéricos, cantidades no positivas o precios negativos. Si encuentra registros inválidos, detiene la ejecución con un error.

In [0]:
df_ventas.printSchema()
print("Registros de venta:", df_ventas.count())

display(df_ventas.groupBy("id").count().filter("count > 1"))

df_invalidos = df_ventas.filter(
    F.col("id").isNull() | F.col("fecha").isNull() |
    F.col("producto").isNull() | F.col("cantidad").isNull() |
    F.col("precio").isNull() | F.isnan("precio") |
    (F.col("cantidad") <= 0) | (F.col("precio") < 0)
)
display(df_invalidos)
assert df_invalidos.count() == 0, "Revisar los registros antes de continuar."

root
 |-- id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)

Registros de venta: 100


id,count


id,fecha,producto,cantidad,precio


Guarda el DataFrame df_ventas como una tabla Delta llamada ventas, sobrescribiendo los datos existentes si la tabla ya existe.

In [0]:
df_ventas.write.format("delta").mode("overwrite").saveAsTable("ventas")


Muestra los detalles técnicos de la tabla ventas, como su formato, ubicación de almacenamiento, tamaño, número de archivos y propiedades de Delta.

In [0]:
%sql
DESCRIBE DETAIL ventas;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,170135fc-926d-4830-9b9c-b3281b498c9d,analitica.data.ventas,null,,2026-10-17T20:20:21.411Z,2026-10-17T20:55:22.000Z,List(),List(),1,2392,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> ff9ba7c2-6a42-4d74-8201-2feab1b8d499)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


Agrupa las ventas por producto, calcula las unidades vendidas y el importe total redondeado a dos decimales, y ordena los productos de mayor a menor importe total.

In [0]:
%sql
SELECT producto, 
sum(cantidad) as unidades,
round(SUM(cantidad * precio),2) AS importe_total
FROM ventas
GROUP BY producto
ORDER BY importe_total DESC

producto,unidades,importe_total
Monitor,64,12799.36
Impresora,85,7641.5
Altavoces,137,6165.0
Disco Duro,77,6159.23
Microfono,93,5579.07
Webcam,137,5478.63
Teclado,130,3378.7
Auriculares,92,2759.08
Mouse,88,1364.0
USB,99,989.01


Databricks visualization. Run in Databricks to view.

Muestra el historial de cambios de la tabla Delta ventas, incluyendo versiones, fechas, operaciones realizadas y usuarios responsables cuando esa información está disponible.

In [0]:
%sql
DESCRIBE HISTORY ventas;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-10-17T20:55:22.000Z,77538316313781,andressabogal315@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""ff9ba7c2-6a42-4d74-8201-2feab1b8d499""}, statsOnLoad -> true)",null,List(3412372199736854),fba06089-a31e-41e8-a53b-62e6655da839,1017-190227-syaso9ee-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2392, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 2392)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-17T20:20:23.000Z,77538316313781,andressabogal315@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""ff9ba7c2-6a42-4d74-8201-2feab1b8d499""}, statsOnLoad -> true)",null,List(3412372199736854),92e6eb9a-97f7-4f00-81da-09c64d9be088,1017-190227-syaso9ee-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 2392)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


Define Spain como país seleccionado y muestra del DataFrame df_enriquecido únicamente los registros correspondientes a ese país.

In [0]:
pais_elegido = "Spain"

display(df_enriquecido.filter(F.col("country") == pais_elegido))

product,order_id,order_date,country,category,units_sold,unit_price,importe,product_family,supplier
Laptop,1,2024-01-02,Spain,Electronics,2,1200,2400,Computers,TechCorp
Mouse,2,2024-01-03,Spain,Accessories,5,25,125,Peripherals,InputMakers
Monitor,5,2024-01-07,Spain,Electronics,2,300,600,Displays,ScreenWorld
Keyboard,8,2024-01-11,Spain,Accessories,4,42,168,Peripherals,InputMakers


Crea o reemplaza la tabla Delta ventas_delta_demo copiando todos los registros y columnas de la tabla ventas.

In [0]:
%sql
CREATE OR REPLACE TABLE ventas_delta_demo USING DELTA AS SELECT * FROM ventas


num_affected_rows,num_inserted_rows


Consulta el historial de ventas_delta_demo, obtiene el número de su versión más reciente y lo guarda en version_inicial como referencia para comparar cambios posteriores.

In [0]:
historial = spark.sql("DESCRIBE HISTORY ventas_delta_demo")
version_inicial = historial.orderBy(F.desc("version")).first()["version"]

print("Versión de referencia:", version_inicial)

Versión de referencia: 0


Actualiza la tabla ventas_delta_demo, aumentando en 1 el precio del registro cuyo id es 1 y redondeándolo a dos decimales.

In [0]:
%sql
UPDATE ventas_delta_demo SET precio = ROUND(precio + 1, 2) WHERE id = 1

num_affected_rows
1


Consulta y muestra todos los datos del registro con id = 1 en ventas_delta_demo para verificar el precio actualizado.

In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id = 1

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,26.99


Inserta un nuevo registro en ventas_delta_demo con ID 1001, fecha 2025-01-31, producto Producto demo, cantidad 2 y precio 10.0.

In [0]:
%sql

INSERT INTO ventas_delta_demo VALUES (1001, DATE '2025-01-31', 'Producto demo', 2, 10.0)

num_affected_rows,num_inserted_rows
1,1


Cuenta todos los registros de la tabla ventas_delta_demo y muestra el total con el nombre registros.

In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo

registros
101


Elimina de la tabla ventas_delta_demo el registro cuyo id es 1001.

In [0]:
%sql 
DELETE FROM ventas_delta_demo WHERE id = 1001

num_affected_rows
1


Cuenta los registros actuales de ventas_delta_demo y muestra el total para comprobar que el registro eliminado ya no está en la tabla.

In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo

registros
100


Crea o reemplaza la vista temporal cambios_ventas con dos registros de ejemplo, incluyendo sus IDs, fechas, productos, cantidades y precios, para utilizarlos en operaciones posteriores.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cambios_ventas AS
SELECT * FROM VALUES
  (1, DATE '2025-01-03', 'Teclado', 20, CAST(27.99 AS DOUBLE)),
  (1001, DATE '2025-01-31', 'Producto demo', 2, CAST(10.0 AS DOUBLE))
AS cambios(id, fecha, producto, cantidad, precio)

Muestra todos los registros de la vista temporal cambios_ventas para consultar los dos registros de ejemplo creados.

In [0]:
%sql
SELECT * FROM cambios_ventas

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99
1001,2025-01-31,Producto demo,2,10.0


Sincroniza ventas_delta_demo con cambios_ventas: actualiza los registros cuyo id coincide e inserta los registros nuevos cuyo id no existe en la tabla destino.

In [0]:
%sql
MERGE INTO ventas_delta_demo AS destino
USING cambios_ventas AS fuente
ON destino.id = fuente.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


Consulta los registros con ID 1 y 1001 de ventas_delta_demo y los ordena por ID para verificar los cambios realizados con el MERGE.

In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id IN (1, 1001) ORDER BY id

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99
1001,2025-01-31,Producto demo,2,10.0


Muestra el historial de versiones y operaciones realizadas en ventas_delta_demo, incluyendo los cambios efectuados mediante UPDATE, INSERT, DELETE y MERGE.

In [0]:
%sql
DESCRIBE HISTORY ventas_delta_demo

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-10-17T20:58:09.000Z,77538316313781,andressabogal315@gmail.com,MERGE,"Map(predicate -> [""(id#17079 = id#17064)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3412372199736854),e51ddc9c-5d1b-4b3b-b893-28672faf7516,1017-190227-syaso9ee-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3293, numTargetBytesRemoved -> 1660, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3465, materializeSourceTimeMs -> 269, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1669, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1461)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
3,2026-10-17T20:56:54.000Z,77538316313781,andressabogal315@gmail.com,DELETE,"Map(predicate -> [""(id#16768 = 1001)""])",null,List(3412372199736854),3936e3f2-97c9-4fe5-b833-c74196d8d197,1017-190227-syaso9ee-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1663, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1210, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1085, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 124)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
2,2026-10-17T20:56:19.000Z,77538316313781,andressabogal315@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3412372199736854),37b11227-19ff-4696-bea1-b539afec02fe,1017-190227-syaso9ee-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1663)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-17T20:55:59.000Z,77538316313781,andressabogal315@gmail.com,UPDATE,"Map(predicate -> [""(id#15966 = 1)""])",null,List(3412372199736854),d28af2bc-709b-414a-b058-af32b88895e7,1017-190227-syaso9ee-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3687, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1798, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1660, rewriteTimeMs -> 1855)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-17T20:55:37.000Z,77538316313781,andressabogal315@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""cd7f73bf-e780-4dcc-a5fa-50bcb6282957""}, statsOnLoad -> true)",null,List(3412372199736854),9b087f9b-c06f-42b9-9aaf-74d49ca2bbc2,1017-190227-syaso9ee-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 2392)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


Recupera la versión inicial de ventas_delta_demo usando version_inicial, compara la cantidad de filas iniciales y actuales, y muestra el registro con ID 1 en ambas versiones para identificar los cambios realizados.

In [0]:
df_inicial = spark.read.option("versionAsOf", version_inicial).table("ventas_delta_demo")

print("Filas iniciales:", df_inicial.count())
print("Filas actuales:", spark.table("ventas_delta_demo").count())
display(df_inicial.filter(F.col("id") == 1))
display(spark.table("ventas_delta_demo").filter(F.col("id") == 1))

Filas iniciales: 100
Filas actuales: 101


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99


Guarda el DataFrame df_enriquecido como tabla Delta pedidos_enriquecido, sobrescribiendo completamente la tabla si ya existe.

In [0]:
df_enriquecido.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pedidos_enriquecido")

Cuenta el total de registros de la tabla pedidos_enriquecido y muestra el resultado con el nombre registros.

In [0]:
%sql
SELECT COUNT(*) AS registros
FROM pedidos_enriquecido;

registros
10


Cuenta los registros de las tablas pedidos_enriquecido, ventas y ventas_delta_demo, y combina los resultados en una sola tabla para comparar sus cantidades.

In [0]:
%sql
SELECT 'Pedidos enriquecidos' AS tabla, COUNT(*) AS registros
FROM pedidos_enriquecido

UNION ALL

SELECT 'Ventas base', COUNT(*)
FROM ventas

UNION ALL

SELECT 'Ensayo Delta', COUNT(*)
FROM ventas_delta_demo;

tabla,registros
Pedidos enriquecidos,10
Ventas base,100
Ensayo Delta,101


Crea o reemplaza la vista v_ventas_por_producto, que agrupa las ventas por producto y calcula las unidades vendidas y el importe total de cada producto.

In [0]:
%sql
CREATE OR REPLACE VIEW v_ventas_por_producto AS
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM ventas
GROUP BY producto


Consulta la vista v_ventas_por_producto y muestra los productos ordenados de mayor a menor según su importe total de ventas.

In [0]:
%sql
SELECT * FROM v_ventas_por_producto ORDER BY importe_total DESC

producto,unidades,importe_total
Monitor,64,12799.36
Impresora,85,7641.5
Altavoces,137,6165.0
Disco Duro,77,6159.23
Microfono,93,5579.07
Webcam,137,5478.63
Teclado,130,3378.7
Auriculares,92,2759.08
Mouse,88,1364.0
USB,99,989.01


Crea o reemplaza la función es_venta_alta, que recibe un importe y devuelve TRUE si es mayor que 500 o FALSE si es menor o igual a 500.

In [0]:
%sql
CREATE OR REPLACE FUNCTION es_venta_alta(importe DOUBLE)
RETURNS BOOLEAN
RETURN importe > 500

Muestra los primeros 10 registros de ventas, calcula el importe de cada venta (cantidad * precio) y utiliza es_venta_alta para indicar si supera los 500.

In [0]:
%sql
SELECT id, producto, cantidad, precio,
       es_venta_alta(cantidad * precio) AS supera_umbral
FROM ventas
ORDER BY id
LIMIT 10

id,producto,cantidad,precio,supera_umbral
1,Teclado,20,25.99,true
2,Webcam,1,39.99,false
3,Auriculares,18,29.99,true
4,Webcam,10,39.99,false
5,Mouse,2,15.5,false
6,Altavoces,13,45.0,true
7,Teclado,11,25.99,false
8,Auriculares,5,29.99,false
9,Monitor,2,199.99,false
10,Microfono,20,59.99,true


Muestra información detallada de la vista v_ventas_por_producto, incluyendo su esquema, tipo de objeto, propiedades y definición SQL.

In [0]:
%sql
DESCRIBE TABLE EXTENDED v_ventas_por_producto

col_name,data_type,comment
producto,string,null
unidades,bigint,null
importe_total,double,null
,,
# Detailed Table Information,,
Catalog,analitica,
Database,data,
Table,v_ventas_por_producto,
Owner,andressabogal315@gmail.com,
Created Time,Sat Oct 17 21:06:53 UTC 2026,


Muestra los permisos asignados sobre la vista v_ventas_por_producto, indicando qué usuarios o grupos pueden acceder a ella y qué acciones tienen autorizadas.

In [0]:
%sql
SHOW GRANTS ON VIEW v_ventas_por_producto

Principal,ActionType,ObjectType,ObjectKey


Los comandos comentados muestran cómo otorgar al grupo lectores_clase permisos para usar el catálogo y esquema y consultar la vista; el comando final muestra los permisos actuales de v_ventas_por_producto sin modificarlos.

In [0]:
%sql
-- GRANT USE CATALOG ON CATALOG analitica TO `lectores_clase`;
-- GRANT USE SCHEMA ON SCHEMA analitica.data TO `lectores_clase`;
-- GRANT SELECT ON VIEW analitica.data.v_ventas_por_producto TO `lectores_clase`;

SHOW GRANTS ON VIEW v_ventas_por_producto;

Principal,ActionType,ObjectType,ObjectKey


Consulta y muestra todos los registros de la vista analitica.data.v_ventas_por_producto, que contiene las unidades vendidas y el importe total agrupados por producto.

In [0]:
%sql
SELECT * FROM analitica.data.v_ventas_por_producto

producto,unidades,importe_total
Teclado,130,3378.7
Webcam,137,5478.63
Auriculares,92,2759.08
Mouse,88,1364.0
Altavoces,137,6165.0
Monitor,64,12799.36
Microfono,93,5579.07
USB,99,989.01
Impresora,85,7641.5
Disco Duro,77,6159.23


# Preguntas para revisar lo aprendido

- #### Si el CSV está en un volumen, ¿qué falta para que otro notebook consulte una tabla persistente?

En Databricks, hay que guardar los datos como una tabla persistente en Unity Catalog mediante saveAsTable, para que otros notebooks puedan consultarla.

- #### Si una unión duplica el importe total, ¿qué comprobación haríamos sobre las llaves?

Revisar que las llaves sean únicas en ambas tablas y que no existan duplicados que multipliquen los registros durante el JOIN.

- #### Si una celda define un filtro, ¿significa que Spark ya calculó todo el resultado?

No. Spark usa evaluación perezosa (lazy evaluation); el filtro se ejecuta cuando una acción, como display() o count(), requiere el resultado.

- #### Si una persona tiene SELECT sobre una tabla, ¿qué otros permisos de uso necesita?

Necesita USE CATALOG, USE SCHEMA y SELECT sobre la tabla o vista, según los permisos de Unity Catalog.

- #### Si el historial de la tabla muestra una versión antigua, ¿basta para garantizar su recuperación?

No. En Databricks, Delta Lake permite consultar versiones anteriores con Time Travel, pero depende de la retención de datos y archivos del historial; VACUUM puede eliminar archivos necesarios para recuperar versiones antiguas.

- #### Si repetimos un append con el mismo archivo, ¿qué puede pasar con las ventas?

Se pueden duplicar las ventas, porque append agrega los registros nuevamente sin comprobar si ya existen. En Databricks, se puede usar MERGE para actualizar o insertar registros según una llave única.
